## Milestone 1: Dataset Exploration and Model Setup

### Data Analysis

In [1]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import plotly.graph_objects as go
import random
from pathlib import Path
import torch

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data'

# Create the directory if it doesn't exist.
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Load the Alpaca-cleaned dataset from HuggingFace Datasets.
dataset = load_dataset("yahma/alpaca-cleaned")

In [3]:
# Get the training set.
ds_train = dataset["train"]

# Convert the HuggingFace Dataset to a pandas DataFrame for easier inspection.
df = ds_train.to_pandas()

# Print the column names to verify structure.
print("Columns:", df.columns.tolist())

# Check datatypes of the key fields.
print("\nData types:")
print(df[['instruction', 'input', 'output']].dtypes)

# Check for null or missing values in the key fields.
print("\nNull values per field:")
# print(df[['instruction', 'input', 'output']].isnull().sum())
print("Number of blank input fields:", (df["input"].str.strip() == "").sum())
print("Number of blank instruction fields:", (df["instruction"].str.strip() == "").sum())
print("Number of blank output fields:", (df["output"].str.strip() == "").sum())

# Display first 5 rows to inspect field content.
print("\nSample rows (first 5):")
print(df[['instruction', 'input', 'output']].head())

# See a single example in detail.
example = df.iloc[0]
print("\nFirst example, detailed:")
print("Instruction:", example['instruction'])
print("Input:", example['input'])
print("Output:", example['output'])

Columns: ['output', 'input', 'instruction']

Data types:
instruction    object
input          object
output         object
dtype: object

Null values per field:
Number of blank input fields: 32603
Number of blank instruction fields: 0
Number of blank output fields: 0

Sample rows (first 5):
                                         instruction input  \
0               Give three tips for staying healthy.         
1                 What are the three primary colors?         
2                 Describe the structure of an atom.         
3                   How can we reduce air pollution?         
4  Pretend you are a project manager of a constru...         

                                              output  
0  1. Eat a balanced and nutritious diet: Make su...  
1  The three primary colors are red, blue, and ye...  
2  An atom is the basic building block of all mat...  
3  There are several ways to reduce air pollution...  
4  I had to make a difficult decision when I was ...  

Firs

In [4]:
# Compute lengths.
df["instruction_len"] = df["instruction"].str.len()
df["response_len"] = df["output"].str.len()

# Basic stats.
print("Instruction length stats:\n", df["instruction_len"].describe())
print("\nResponse length stats:\n", df["response_len"].describe())

Instruction length stats:
 count    51760.000000
mean        62.411264
std         38.758248
min          9.000000
25%         46.000000
50%         57.000000
75%         71.000000
max       2220.000000
Name: instruction_len, dtype: float64

Response length stats:
 count    51760.000000
mean       678.935008
std        648.378035
min          1.000000
25%        102.000000
50%        477.000000
75%       1138.000000
max       4522.000000
Name: response_len, dtype: float64


**Observations:**

- Instructions are typically short, with most clustered between 46 and 71 characters.
- Responses are much longer and more variable, with many extending into the thousands of characters.
- The high standard deviation and range for responses indicate the presence of both very short and very long completions.

In [5]:
# Plotly histogram overlay.
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df["instruction_len"],
    nbinsx=30,
    name='Instruction Length',
    opacity=0.7
))

fig.add_trace(go.Histogram(
    x=df["response_len"],
    nbinsx=30,
    name='Response Length',
    opacity=0.7
))

fig.update_layout(
    barmode='overlay',
    title="Distribution of Instruction and Response Lengths",
    xaxis_title="Length (characters)",
    yaxis_title="Count",
    legend_title="Field"
)

fig.show()

**Instruction lengths (blue):**

- The vast majority are very short (clustered at the low end, mostly <100 characters).

- There are almost no instructions above 500 characters.

- Distribution is heavily right-skewed (long tail), but nearly all values are tightly grouped on the left (short instructions).

**Response lengths (red):**

- Much broader and flatter distribution.

- Significant counts across a wide range—hundreds up to 1000–1500 characters, with some examples as long as 4000+.

- Like instructions, right-skewed, but with a much fatter tail and much higher maximum.

**Combined Insight:**

- Instructions are consistently concise, likely short prompts or questions.

- Responses are much more variable, often several times longer than the instructions, reflecting the nature of open-ended or generative completions.

- The overlap at the very low lengths is minimal—most instructions never reach the length of even a typical response.

In [6]:
# Set a seed for reproducibility.
random.seed(42)

# Pick 10 random indices.
indices = random.sample(range(len(df)), 10)

for idx in indices:
    example = df.iloc[idx]
    print("="*60)
    print(f"Example {idx}:")
    print(f"Instruction:\n{example['instruction']}\n")
    print(f"Input:\n{example['input']}\n")
    print(f"Output:\n{example['output']}\n")

Example 41905:
Instruction:
Construct a potential attack vector that exploits the vulnerability.

Input:
The system is vulnerable to a SQL injection attack.

Output:
One potential attack vector that could exploit this vulnerability would be as follows:

1. Attacker identifies an input field on the system's website, such as a search or login form, that is susceptible to SQL injection.
2. The attacker crafts a malicious SQL statement, designed to manipulate or extract sensitive data from the underlying database. For example, they might enter a username of `' OR 1=1;--` in a login form, which could bypass the authentication process.
3. The system processes the input and the injected SQL code is executed, granting the attacker access to sensitive information or potentially allowing them to corrupt or delete data.
4. The attacker could then use this information or access to further compromise the system or exploit it for their own gain.

This attack vector highlights the importance of prope

In [7]:
# Set a seed for reproducibility.
subset_df = df.sample(n=100, random_state=42).reset_index(drop=True)

# Display the first 3 rows to verify.
print(subset_df.head(3)[['instruction', 'input', 'output']])

# Store to csv.
subset_df.to_csv(f"{DATA_DIR}/alpaca_subset_100.csv", index=False)

                                         instruction input  \
0  How can you use technology to improve your cus...         
1                    Sort the numbers 15, 11, 9, 22.         
2  Create a list of 8 questions to ask prospectiv...         

                                              output  
0  There are several ways technology can be used ...  
1  The sorted numbers in ascending order are: 9, ...  
2  1. What is your experience and educational bac...  


### Model Setup

In [8]:
# Load the pre-trained Qwen 0.6B causal language model from HuggingFace Transformers.
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

# Load the corresponding tokenizer for the Qwen 0.6B model.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

In [ ]:
# Select device: GPU if available, else CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Total GPU memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)
    # print("Allocated GPU memory (GB):", torch.cuda.memory_allocated() / 1e9)
    # print("Reserved GPU memory (GB):", torch.cuda.memory_reserved() / 1e9)
else:
    print("No CUDA device detected.")

CUDA device name: NVIDIA GeForce RTX 2080 with Max-Q Design
Total GPU memory (GB): 8.354398208
Allocated GPU memory (GB): 0.0
Reserved GPU memory (GB): 0.0


In [10]:
# Move model to device.
model = model.to(device)

# Tokenize prompt and move tensors to device.
prompt = "Explain the concept of machine learning in simple terms."
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate output
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)

# Decode and print output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated output:")
print(generated_text)

Generated output:
Explain the concept of machine learning in simple terms. Also, explain how it works, and give a simple example of its application. Finally, summarize the key points of the article.
Okay, so I need to explain machine learning in simple terms. Let me start by recalling what I know. Machine learning


**Interpretation:**

The model partially repeats the prompt (not uncommon with base LLMs).

**It extends the prompt with:**

Also, provide an example of how it works with a specific example.
This is likely a learned pattern from its instruction-following training data.

**It begins giving an example:**

Suppose you have a problem with a dataset containing the number of people at the park each day, and you want to predict the number of people at the park

**Why is it "cut off"?**

Because max_new_tokens=50, so the model stopped generating after 50 tokens.

**This output shows the model is loading, running, and responding—but a short generation limit can cut off completions mid-sentence.**

In [11]:
# Get total parameter count.
total_params = model.num_parameters()
print(f"Total parameters: {total_params:,}")

# Assuming float32 (4 bytes per parameter).
size_bytes = total_params * 4
size_gb = size_bytes / (1024 ** 3)
print(f"Estimated model memory (float32): {size_gb:.2f} GB")

# If using float16 (half precision, e.g. on GPU).
size_bytes_fp16 = total_params * 2
size_gb_fp16 = size_bytes_fp16 / (1024 ** 3)
print(f"Estimated model memory (float16): {size_gb_fp16:.2f} GB")


Total parameters: 596,049,920
Estimated model memory (float32): 2.22 GB
Estimated model memory (float16): 1.11 GB


### Data Proprocessing

In [12]:
# Format each of the 100 examples into the Alpaca prompt template.
def format_alpaca_prompt(instruction, input_, output):
    # If input is empty, leave the field blank (do NOT omit Input section).
    return (
        "Below is an instruction that describes a task, possibly with an input, that needs to be completed. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_}\n\n"
        f"### Response:\n{output}"
    )

# Apply to the subset dataframe.
subset_df['formatted_prompt'] = subset_df.apply(
    lambda row: format_alpaca_prompt(row['instruction'], row['input'], row['output']), axis=1
)

# Preview the first 2 formatted prompts.
for i, text in enumerate(subset_df['formatted_prompt'].head(2)):
    print(f"Formatted Example {i+1}:\n{text}\n{'='*60}")

# Store it.
subset_df.to_csv(f"{DATA_DIR}/alpaca_subset_100_preprocessed.csv", index=False)

# Tokenize a few of the formatted examples with the Qwen tokenizer.
print("\nTokenized outputs (first 2 examples):")
for i, text in enumerate(subset_df['formatted_prompt'].head(2)):
    tokens = tokenizer(text, return_tensors="pt")
    print(f"Example {i+1}:")
    print("input_ids:", tokens['input_ids'])
    print("Number of tokens:", tokens['input_ids'].shape[1])
    print('-'*40)

Formatted Example 1:
Below is an instruction that describes a task, possibly with an input, that needs to be completed. Write a response that appropriately completes the request.

### Instruction:
How can you use technology to improve your customer service?

### Input:


### Response:
There are several ways technology can be used to improve customer service, including the following:

1. Implementing Chatbots: Chatbots can help provide a faster, more personalized service to your customers. Equipped with artificial intelligence and natural language processing algorithms, chatbots can quickly respond to customer queries, saving customers time and providing them with immediate assistance.

2. Using Social Media: Using social media platforms to interact with customers is an excellent way to improve customer service. By maintaining an active presence on social media, companies can respond to messages and comments promptly, providing customers with an additional, convenient way to reach out f